# Demo 04 — Learning Taxonomies (D11)

**Class 01 · Block 5** (slides: `slides/01_foundations.md`)

One dataset (handwritten digits, 8×8 pixels), four ways of learning from it, plus a small reinforcement learning world.
What changes is **where the learning signal comes from**.

| Setting | Signal | Course topics |
|---|---|---|
| Supervised | human labels for every sample | 3–6 |
| Unsupervised | the structure of the data alone | 7–8 |
| Semi-supervised | a few labels + many unlabelled samples | 9 |
| Self-supervised | labels created from the data itself | 9 |
| Reinforcement | rewards obtained by acting in an environment | 10 |

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import adjusted_rand_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.semi_supervised import LabelSpreading, SelfTrainingClassifier

SEED = 42
rng = np.random.default_rng(SEED)

X, y = load_digits(return_X_y=True)
X = X / 16.0
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=SEED)
print(f"{len(X_train)} training images, {len(X_test)} test images, 64 pixels each")

fig, axes = plt.subplots(1, 10, figsize=(12, 1.6))
for digit, ax in enumerate(axes):
    ax.imshow(X_train[y_train == digit][0].reshape(8, 8), cmap="gray_r")
    ax.set(title=str(digit), xticks=[], yticks=[])
plt.show()

## 1. Supervised learning — every image has a label

Learn $f: \text{image} \rightarrow \text{digit}$ from $(x, y)$ pairs.

In [ ]:
def make_classifier():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))


supervised = make_classifier().fit(X_train, y_train)
acc_supervised = supervised.score(X_test, y_test)
print(f"Supervised, {len(y_train)} labels: test accuracy = {acc_supervised:.3f}")

## 2. Unsupervised learning — no labels at all

Ask $k$-means to group the images into 10 clusters. It never sees a label; we only use labels **afterwards** to check the result.

In [ ]:
kmeans = KMeans(n_clusters=10, n_init=10, random_state=SEED).fit(X_train)
print(f"Agreement between clusters and true digits (adjusted Rand index, 0 = random, 1 = perfect): "
      f"{adjusted_rand_score(y_train, kmeans.labels_):.3f}")

fig, axes = plt.subplots(1, 10, figsize=(12, 1.6))
for c, ax in enumerate(axes):
    ax.imshow(kmeans.cluster_centers_[c].reshape(8, 8), cmap="gray_r")
    ax.set(title=f"cluster {c}", xticks=[], yticks=[])
plt.suptitle("Cluster centres found without labels", y=1.15)
plt.show()

X_2d = PCA(n_components=2, random_state=SEED).fit_transform(X_train)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(X_2d[:, 0], X_2d[:, 1], c=kmeans.labels_, cmap="tab10", s=8)
axes[0].set_title("what k-means found (no labels)")
axes[1].scatter(X_2d[:, 0], X_2d[:, 1], c=y_train, cmap="tab10", s=8)
axes[1].set_title("true digits (hidden from the algorithm)")
plt.show()

## 3. Semi-supervised learning — only a few labels

Keep labels for only **10% of the images of each digit** and hide the rest (label `-1`).

In [ ]:
LABELLED_FRACTION = 0.10
labelled = np.concatenate([
    rng.choice(idx, round(LABELLED_FRACTION * len(idx)), replace=False)
    for idx in (np.flatnonzero(y_train == d) for d in range(10))
])
y_partial = np.full_like(y_train, -1)
y_partial[labelled] = y_train[labelled]
print(f"Labelled: {len(labelled)}   Unlabelled: {(y_partial == -1).sum()}")

only_labelled = make_classifier().fit(X_train[labelled], y_train[labelled])
self_training = SelfTrainingClassifier(make_classifier(), threshold=0.9).fit(X_train, y_partial)
spreading = LabelSpreading(kernel="knn", n_neighbors=7).fit(X_train, y_partial)

print(f"\n{'method':50s}{'test accuracy':>14s}")
n_lab = len(labelled)
print(f"{f'supervised, {n_lab} labels only':50s}{only_labelled.score(X_test, y_test):14.3f}")
print(f"{f'self-training (pseudo-labelling), {n_lab} labels':50s}{self_training.score(X_test, y_test):14.3f}")
print(f"{f'label spreading (similarity graph), {n_lab} labels':50s}{spreading.score(X_test, y_test):14.3f}")
print(f"{'supervised, all labels (reference)':50s}{acc_supervised:14.3f}")

Unlabelled data is cheap, and the *structure* it reveals (similar images are close together) lets a few labels go much further.

## 4. Self-supervised learning — the data labels itself

**Pretext task:** hide the bottom half of each image and predict it from the top half. The "labels" (the hidden pixels) come for free, so no human annotation is needed.
The same idea, at a much larger scale, trains AutoEncoders, language models (predict the masked or next word), and JEPA models (Topic 9).

In [ ]:
top, bottom = X_train[:, :32], X_train[:, 32:]
inpainter = Ridge(alpha=1.0).fit(top, bottom)
predicted_bottom = inpainter.predict(X_test[:, :32])

fig, axes = plt.subplots(3, 8, figsize=(10, 4))
for i in range(8):
    masked = X_test[i].copy()
    masked[32:] = 0.5
    completed = np.concatenate([X_test[i, :32], predicted_bottom[i]])
    for row, img in enumerate([X_test[i], masked, completed]):
        axes[row, i].imshow(img.reshape(8, 8), cmap="gray_r", vmin=0, vmax=1)
        axes[row, i].set(xticks=[], yticks=[])
for row, name in enumerate(["original", "masked", "predicted"]):
    axes[row, 0].set_ylabel(name)
plt.suptitle("Self-supervised pretext task: complete the digit (no labels used)")
plt.show()

To predict the missing half, the model must capture how digits are **shaped**. That knowledge is a useful representation for later tasks.

## 5. Reinforcement learning — learning from rewards

There is no dataset at all. An **agent** acts in an **environment** and receives a **reward**:

* a 5×5 grid; the agent starts at the top-left corner;
* actions: up, down, left, right;
* reaching the **goal** (bottom-right) gives **+1**; falling into a **pit** gives **−1**; every other step costs **−0.01**.

**Q-learning** keeps a table $Q(s, a)$: "how good is action $a$ in state $s$". After each step it nudges the estimate towards the reward plus the value of the next state:

$$Q(s,a) \leftarrow Q(s,a) + \alpha \big[\, r + \gamma \max_{a'} Q(s', a') - Q(s,a) \,\big]$$

With probability $\varepsilon$ the agent **explores** (random action); otherwise it **exploits** its best known action.

In [ ]:
SIZE, START, GOAL = 5, (0, 0), (4, 4)
PITS = {(1, 1), (2, 3), (3, 1)}
MOVES = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # up, down, left, right
ARROWS = ["↑", "↓", "←", "→"]


def step(state, action):
    r, c = state
    dr, dc = MOVES[action]
    nxt = (min(max(r + dr, 0), SIZE - 1), min(max(c + dc, 0), SIZE - 1))
    if nxt == GOAL:
        return nxt, 1.0, True
    if nxt in PITS:
        return nxt, -1.0, True
    return nxt, -0.01, False


Q = np.zeros((SIZE, SIZE, 4))
alpha, gamma, epsilon = 0.5, 0.95, 0.2
returns = []
for episode in range(500):
    state, total, done, steps = START, 0.0, False, 0
    while not done and steps < 100:
        action = rng.integers(4) if rng.uniform() < epsilon else int(np.argmax(Q[state]))
        nxt, reward, done = step(state, action)
        target = reward if done else reward + gamma * Q[nxt].max()
        Q[state][action] += alpha * (target - Q[state][action])
        state, total, steps = nxt, total + reward, steps + 1
    returns.append(total)

returns = np.array(returns)
print(f"Average return, first 50 episodes: {returns[:50].mean():+.3f}")
print(f"Average return, last 50 episodes : {returns[-50:].mean():+.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
window = 20
axes[0].plot(np.convolve(returns, np.ones(window) / window, mode="valid"))
axes[0].set(xlabel="episode", ylabel=f"return (moving average of {window})", title="The agent improves from rewards alone")

ax = axes[1]
ax.set(xlim=(-0.5, SIZE - 0.5), ylim=(SIZE - 0.5, -0.5), xticks=[], yticks=[], title="Learned policy (best action per cell)")
for r in range(SIZE):
    for c in range(SIZE):
        if (r, c) in PITS:
            ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1, color="tab:red", alpha=0.6))
            ax.text(c, r, "pit", ha="center", va="center")
        elif (r, c) == GOAL:
            ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1, color="tab:green", alpha=0.6))
            ax.text(c, r, "goal", ha="center", va="center")
        else:
            ax.text(c, r, ARROWS[int(np.argmax(Q[r, c]))], ha="center", va="center", fontsize=18)
for k in range(SIZE + 1):
    ax.axhline(k - 0.5, color="gray", lw=0.5)
    ax.axvline(k - 0.5, color="gray", lw=0.5)
plt.tight_layout()
plt.show()

Nobody told the agent which moves are correct. It discovered a path around the pits **by trial and error**, guided only by rewards.
Reinforcement learning is expanded in Topic 10.